<a href="https://colab.research.google.com/github/jimhopgtu/google-ai-agents-daily-content-tool/blob/main/Daily_Relevant_Content.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools.google_search_tool import GoogleSearchTool
from google.adk.tools import FunctionTool
from google.genai import types
import json
from pydantic import BaseModel, Field
from typing import List
import os
from google.colab import userdata, drive
from datetime import datetime
import pytz
eastern_tz = pytz.timezone('America/New_York')

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [14]:

# This pulls the secret you just created and sets it as an environment variable
# Most Google SDKs (including ADK) look for "GOOGLE_API_KEY" automatically.
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

print("✅ API Key successfully loaded into environment!")

# from IPython.core.display import display, HTML
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)


# Ensure Drive is mounted correctly
drive.mount('/content/drive', force_remount=True)

print("Executed at:", datetime.now(eastern_tz))

✅ API Key successfully loaded into environment!
Mounted at /content/drive
Executed at: 2025-12-22 08:43:41.221443-05:00


In [19]:
search_instruction = """
You are a Research Scout for an Analytics Leader.
Your goal is to provide a balanced mix of content . For every run, you MUST use the search tool to find:

1. THE LATEST (24h): Top 3 industry-shifting news (e.g., Anthropic, OpenAI, Google).
2. THE ARCHITECTURE: Top 1 technical blog posts from engineering-heavy companies
   (e.g., MongoDB, Pinecone, Meta Engineering) that discuss 'why' or 'how'—not just 'what'.
3. THE STACK: Top 2 recent updates from the modern BI stack (e.g., dbt, Snowflake, Databricks, BigQuery, Looker, Atlan, PowerBI, Tableau)
4. THE LOCAL: 1 AI event in the NYC area or a major remote global summit.
5. Top 1 Obsidian plug in or use case that are new and could be useful.

## CRITICAL CONSTRAINTS
<Constraints>
- ANTI-FLUFF: If results are 'marketing fluff', refine keywords to include 'technical deep dive'.
- OUTPUT FORMAT: Return ONLY the URL and a 1-2 sentence summary. No full articles.
- MULTIMEDIA: YouTube videos are acceptable.
</Constraints>

## OUTPUT FORMAT (MANDATORY)
For every single item you find, you MUST follow this exact format:
- **Title**: [Name of the article/video]
- **Source URL**: [Insert the full direct link here]
- **Summary**: [1-2 sentences of why this matters for a data leader]

## CRITICAL RULES
- NEVER provide a news item without a corresponding URL.
- If you find a great story but the URL is missing from your tool output, do not include the story.
"""

# Define the data structure as a Python list/dictionary
LEADER_CONTEXT_DATA = {
    "ANALYTICS_LEADER_CONTEXT": [
        {
            "Category": "Modeling",
            "Shift": "Causal Inference (MMM/MTA), Advanced Models",
            "Action": "Prioritize Experimentation and Causal Strategy (A/B testing, incrementality)"
        },
        {
            "Category": "Architecture",
            "Shift": "Semantic Layer is SOT, often led by **Knowledge Engineer**",
            "Action": "Architect **AI Trust** and robust **Data Governance**"
        },
        {
            "Category": "Role & Skills",
            "Shift": "Analyst as Prompt Engineer/Consultant",
            "Action": "Coach for **Business Acumen** (the 'Why') and focus on recommendations"
        },
        {
            "Category": "Specialization",
            "Shift": "Deep expertise in one major stack (e.g., GCP, Azure)",
            "Action": "Standardize and Optimize the chosen stack to maximize value"
        },
        {
            "Category": "Analyst Profile",
            "Shift": "**Hybrid Role** (Business SME + Data Engineering + **Knowledge Engineering**)",
            "Action": "Redefine career path, mandate **DE fundamentals** and context structuring"
        },
        {
            "Category": "Governance",
            "Shift": "Mandatory focus on **Data Governance** and **AI TRiSM**",
            "Action": "Audit AI outputs, enforce lineage, and ensure ethical compliance"
        },
        {
            "Category": "Speed",
            "Shift": "Shift to **Real-Time** and **Edge Analytics**",
            "Action": "Invest in Modern Architecture (e.g., Data Mesh) for streaming data processing"
        },
        {
            "Category": "Leadership",
            "Shift": "Highest value in Human-Centric/Soft Skills and organizational influence",
            "Action": "Cultivate critical thinking, emotional intelligence, and **cross-department bridge-building** to remove data silos"
        }
    ]
}

# Convert it to a pretty-printed string ONLY when you need to feed it to the Agent
context_string = json.dumps(LEADER_CONTEXT_DATA, indent=2)



relevance_instruction = f"""
## ROLE
You are a Quality Controller for an Analytics Leader.

## YOUR TASK
1. Analyze the articles in 'raw_news_data'.
2. Assign a score (1-5) based on technical depth and strategic relevance.
3. **CRITICAL GATEKEEPER LOGIC**:
   - If an article scores **less than 3**, you MUST mark its status as "REJECTED".
   - Only articles with a score of 3, 4, or 5 should be marked as "APPROVED".
   - Discard "marketing fluff", generic AI hype, or basic tutorials.
4. **GOAL**: We need at least 5 high-quality (Score 3+) articles in total.

## OUTPUT FORMAT
Return a JSON list of objects with these keys: title, url, summary, score, category, action, and status.
"""

from pydantic import BaseModel, Field, AliasChoices
from typing import List, Optional

# This replaces 'ScoredArticle' to match the system expectations
class EvaluatedArticle(BaseModel):
    title: str
    # 'validation_alias' allows the AI to say 'url' OR 'source_url' without crashing
    url: str = Field(validation_alias=AliasChoices('url', 'source_url'))
    summary: str
    score: int
    # Making these 'Optional' with default values prevents "Missing Field" crashes
    category: Optional[str] = "General"
    action: Optional[str] = "Review for strategy"

class RelevanceResponse(BaseModel):
    evaluated_articles: List[EvaluatedArticle]


print("Executed at:", datetime.now(eastern_tz))

Executed at: 2025-12-22 08:46:33.635894-05:00


In [21]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("✅ exit_loop function created.")
print("Executed at:", datetime.now(eastern_tz))

✅ exit_loop function created.
Executed at: 2025-12-22 08:46:45.090085-05:00


In [22]:

# 1. Search Agent
search_agent = Agent(
    name="news_finder",
    model="gemini-2.0-flash", # Use 2.0 for speed/tool use gemini-2.0-flash gemini-1.5-flash or gemini-1.5-flash-8b
    tools=[GoogleSearchTool()],
    instruction=search_instruction,
    output_key="raw_news_data"
)

# 2. Relevance Agent
relevance_agent = Agent(
    name="relevance_evaluator",                  # 1. Added required 'name'
    model=Gemini(model_id="gemini-2.0-flash-exp"),
    instruction=relevance_instruction            # 2. Changed 'instructions' to 'instruction'
)

# 3. Loop Agent
story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[search_agent, relevance_agent],
    # The SDK usually looks for a single condition string referencing the state
    max_iterations=3
)

# 4. Reporting Agent
reporting_agent = Agent(
    name="reporting_agent",
    model="gemini-2.0-flash",
    instruction="""
    ## TASK
    1. Read the articles stored in {{approved_stories?}}.
    2. If the vault is empty, simply state "No highly relevant news found for today."
    3. If articles exist, format them into a professional Markdown report.

    ## FORMAT REQUIREMENTS
    - Use ## Headers for different categories.
    - Use **Bolding** for key takeaways.
    - Ensure every Title is a clickable [Markdown Link](URL).
    - Provide a "Why this matters" section for each article.
    """
)

print("Executed at:", datetime.now(eastern_tz))

Executed at: 2025-12-22 08:46:45.908056-05:00


In [27]:
# =========================================================
# UPDATED CELL 8: AUTOMATED RELEVANCE LOOP & FILTERING
# =========================================================
import re
import json
import pandas as pd

# 1. RESET FOR FRESH RUN
# We clear the vault to ensure the 'while' loop triggers
APPROVED_STORIES_STORAGE["approved_stories"] = {"evaluated_articles": []}

target_count = 5  # Minimum number of Score 3+ articles needed
iteration = 0
max_iterations = 3
seen_urls = set()

print(f"🚀 Starting Research Loop. Goal: {target_count} high-quality articles.")

# 2. START THE REFINEMENT LOOP
while len(APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"]) < target_count and iteration < max_iterations:
    iteration += 1
    current_count = len(APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"])

    # Create a unique session per iteration to force fresh tool usage
    loop_session = f"daily_report_{pd.Timestamp.now().strftime('%H%M%S')}_{iteration}"

    print(f"\n--- 🔄 ITERATION {iteration} (Current Approved: {current_count}/{target_count}) ---")

    # STEP A: SEARCH
    runner = InMemoryRunner(agent=search_agent)
    print("🔍 Search Agent: Finding new articles...")
    search_events = await runner.run_debug(
        f"Find 5-10 unique news articles for an Analytics Leader. Avoid these URLs: {list(seen_urls)[:5]}",
        session_id=loop_session
    )

    # STEP B: RELEVANCE & SCORING
    runner.agent = relevance_agent
    print("⚖️ Relevance Agent: Scoring content and filtering 'fluff'...")
    relevance_events = await runner.run_debug(
        "Analyze the new 'raw_news_data'. Only approve items with score >= 3.",
        session_id=loop_session
    )

    # STEP C: PYTHON-LEVEL FILTERING
    # This acts as the final gate regardless of what the AI says
    raw_ai_text = relevance_events[-1].content.parts[0].text
    clean_json = re.sub(r'^```json\s*|```$', '', raw_ai_text, flags=re.MULTILINE | re.DOTALL).strip()

    try:
        parsed_json = json.loads(clean_json)
        # Handle different potential JSON structures from the AI
        articles = parsed_json if isinstance(parsed_json, list) else parsed_json.get("evaluated_articles", parsed_json.get("articles", []))

        new_approvals = 0
        for item in articles:
            url = item.get("url")
            score = item.get("score", 0)

            # ONLY keep if Score >= 3 AND not a duplicate
            if score >= 3 and url not in seen_urls:
                APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"].append(item)
                seen_urls.add(url)
                new_approvals += 1
            elif score < 3:
                print(f"🗑️ Discarded (Score {score}): {item.get('title')[:50]}...")

        print(f"✅ Added {new_approvals} qualified articles in this iteration.")

    except Exception as e:
        print(f"❌ Error parsing Relevance JSON: {e}")

# 3. FINAL REPORTING
final_vault = APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"]

if len(final_vault) >= 1:
    print(f"\n📊 Final Vault size: {len(final_vault)}. Generating Professional Report...")
    runner.agent = reporting_agent

    # We pass the EXACT filtered JSON to the reporter
    final_report_events = await runner.run_debug(
        f"Generate the markdown report using ONLY this data: {json.dumps({'evaluated_articles': final_vault})}",
        session_id="final_reporting_session"
    )
    print("\n✅ Report Generated Successfully.")
else:
    print("⚠️ Loop ended: Could not find enough high-quality content.")

# --- COST & TOKEN TRACKING ---
total_tokens = 0
for event in (search_events + relevance_events + (final_report_events if 'final_report_events' in locals() else [])):
    if hasattr(event, 'usage_metadata') and event.usage_metadata:
        total_tokens += event.usage_metadata.prompt_token_count
print(f"\n💰 Estimated tokens used: {total_tokens:,}")

🚀 Starting Research Loop. Goal: 5 high-quality articles.

--- 🔄 ITERATION 1 (Current Approved: 0/5) ---
🔍 Search Agent: Finding new articles...

 ### Created new session: daily_report_135205_1

User > Find 5-10 unique news articles for an Analytics Leader. Avoid these URLs: []


news_finder > Okay, I will find 5-10 unique news articles suitable for an Analytics Leader, following the specified categories and format, while avoiding the provided URLs.

- **Title**: OpenAI, Anthropic, and Google Chase Private Data Deals after Exhausting the Internet
- **Source URL**: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFsOle3ucl61gMazaXzbSOaaIK8Ua7JU2GU5jYQB-zbcIbOpfvC7P9LUqkSg9D7CxsFLwqKT8FORk7GkTcdTY_pXqqwCkpL4S2L7WD6SQnfNFFYNWadN3QmgWntDeU0GQzQI6vU2RAc-1Od1IbbyYQuvqtnzM3vpmCN-07V8ecGB4ZuUBmOVdDMOmvc21WWdqJgfEaJTm-leHaRTceG2IrFytpefjvZA1mWTEg=
- **Summary**: OpenAI, Anthropic, and Google are exploring private data deals with companies in biotech, accounting, and healthcare to license specialized datasets for training their AI models, as they have largely exhausted internet data. These deals present legal and privacy challenges but aim to improve AI responses to complex, industry-specific tasks.

- **Title**: Lower-Cost Vector Retrieval with Voyag

news_finder > Here's a summary of the latest news and resources for an Analytics Leader:

*   **Title**: Google and Anthropic approach LLMs differently - Understanding AI
*   **Source URL**: [https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEq3VmK9GK2Zc3b-oZLhs_mnML08PAdMgYpwwYCb74uuAb3ix8HH0AbA_hsqEGYvZA1LQnUVaQ_DrOLbBpzpa-AsOBNFNbmBqBreBxA_HN6bG5iyI2kL7tssKDWnIx43hyEpEd9ytILGHcliKFespQMwO6hUWSdeWxuf7TJ8_Z6](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEq3VmK9GK2Zc3b-oZLhs_mnML08PAdMgYpwwYCb74uuAb3ix8HH0AbA_hsqEGYvZA1LQnUVaQ_DrOLbBpzpa-AsOBNFNbmBqBreBxA_HN6bG5iyI2kL7tssKDWnIx43hyEpEd9ytILGHcliKFespQMwO6hUWSdeWxuf7TJ8_Z6)
*   **Summary**: This article discusses the different approaches Google and Anthropic are taking to build LLMs, highlighting the cultural differences that drive their dramatically different approaches to model building, noting that both models were trained on TPUs.

*   **Title**: OpenAI releases GPT-5.2 to take on Google an

In [28]:
# This cell exports the report to a markdown file in Google Drive

# 1. Setup the Path
today_date = datetime.now(eastern_tz).strftime("%Y-%m-%d_%H%M")
filename = f"{today_date}_Analytics_Report.md"
save_path = "/content/drive/MyDrive/AI/Obsidian Vault/daily news/"

# 2. Create folder if it's missing
if not os.path.exists(save_path):
    print(f"Creating missing directory: {save_path}")
    os.makedirs(save_path, exist_ok=True)

full_path = os.path.join(save_path, filename)

# 3. Extract and Clean the Report
try:
    # Look back for the actual text content
    final_report_text = ""
    for event in reversed(final_report):
        if event.content and event.content.parts:
            text_parts = [p.text for p in event.content.parts if p.text]
            if text_parts:
                final_report_text = "\n".join(text_parts)
                break

    if not final_report_text:
        raise ValueError("No report content found in final_report events")

    # Remove the ```markdown code block wrappers
    clean_report = re.sub(r'^```(?:markdown)?\n?|```$', '', final_report_text.strip(), flags=re.MULTILINE)

    # Add metadata header
    header = f"""---
title: Analytics Leader Daily Report
date: {datetime.now(eastern_tz).strftime("%Y-%m-%d")}
generated: {datetime.now(eastern_tz).strftime("%Y-%m-%d %H:%M %Z")}
---

"""

    final_content = header + clean_report

    # 4. Write the file and FORCE a sync
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(final_content)
        f.flush()  # Force write to the OS buffer
        os.fsync(f.fileno())  # Force write to the disk

    print(f"✅ Success! File physically written.")
    print(f"📂 Filename: {filename}")
    print(f"📍 Full Path: {full_path}")
    print(f"📏 File Size: {len(final_content)} characters")

    # Show first few lines as preview
    print("\n--- PREVIEW (first 500 chars) ---")
    print(final_content[:500])
    if len(final_content) > 500:
        print("...")
    print("--- END PREVIEW ---\n")

except Exception as e:
    print(f"❌ Error during saving: {e}")
    import traceback
    traceback.print_exc()

    # Fallback: Try to save raw data
    print("\n🔄 Attempting fallback save with raw storage data...")
    try:
        if APPROVED_STORIES_STORAGE.get("approved_stories"):
            fallback_content = "# Analytics Report (Fallback)\n\n"
            fallback_content += f"Generated: {datetime.now(eastern_tz).strftime('%Y-%m-%d %H:%M %Z')}\n\n"

            for article in APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"]:
                fallback_content += f"## [{article['title']}]({article['url']})\n\n"
                fallback_content += f"**Score:** {article['score']}/5 | **Category:** {article['category']}\n\n"
                fallback_content += f"{article['summary']}\n\n"
                fallback_content += f"**Action:** {article['action']}\n\n"
                fallback_content += "---\n\n"

            with open(full_path, "w", encoding="utf-8") as f:
                f.write(fallback_content)
                f.flush()
                os.fsync(f.fileno())

            print(f"✅ Fallback save successful!")
            print(f"📂 Filename: {filename}")
    except Exception as fallback_error:
        print(f"❌ Fallback also failed: {fallback_error}")

# 5. Final verification
print("\n🔍 File verification:")
!ls -lh "{full_path}"

print("Executed at:", datetime.now(eastern_tz))

✅ Success! File physically written.
📂 Filename: 2025-12-22_0853_Analytics_Report.md
📍 Full Path: /content/drive/MyDrive/AI/Obsidian Vault/daily news/2025-12-22_0853_Analytics_Report.md
📏 File Size: 10055 characters

--- PREVIEW (first 500 chars) ---
---
title: Analytics Leader Daily Report
date: 2025-12-22
generated: 2025-12-22 08:53 EST
---

## AI & Data Strategy
### [OpenAI, Anthropic, and Google Chase Private Data Deals after Exhausting the Internet](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGQOzIH7hQEHCR0JmRG0m0XKAAZqkQZuIzB_V1snIY-1N8L0DtoN6GjrFcbizhCLjtaf1dkIVBVYEv8bq-e3ZzDo6LZvSEnK2A8GObittZM3fHnkDx4aIYGkJw6StEiRHPnI2Syae72zGRJ73fzZwdRhZyxIZSgA2ER7sB9oCzHuNVSq3XpqPnjYZ6d3fwuTVO9lpWGZ4t1UH-f5WtDtwF7IGBAU-3lY
...
--- END PREVIEW ---


🔍 File verification:
-rw------- 1 root root 9.9K Dec 22 13:53 '/content/drive/MyDrive/AI/Obsidian Vault/daily news/2025-12-22_0853_Analytics_Report.md'
Executed at: 2025-12-22 08:53:54.838851-05:00


In [ ]:
# !rm -rf /content/drive